# MVSR – Vorbereitung Einheit 4
## Image Features, Deskriptoren und Feature Matching

**Bearbeitungszeit:** ca. 30–45 Minuten

Dieses Notebook bereitet die Inhalte der vierten MVSR-Einheit praktisch vor. Im Mittelpunkt stehen **lokale Bildmerkmale**, ihre Beschreibung durch **Deskriptoren** und der Vergleich von Merkmalen zwischen zwei Bildern.

Ausgehend von den Kanten und einfachen Bildmerkmalen der vorherigen Einheit untersuchen Sie zunächst **Harris Corners** und **Good Features to Track**. Anschließend wird der Übergang von einzelnen Eckpunkten zu Deskriptoren betrachtet. Mit **SIFT** werden schließlich skalierungs- und rotationsrobustere Merkmale extrahiert und zwischen zwei realen Bildern verglichen.

Für dieses Binder-Notebook wird **C++17 mit OpenCV 4.6** verwendet.

### Lernziele

Nach Bearbeitung dieses Notebooks können Sie ...

- erklären, warum Ecken häufig eindeutiger als einzelne Kanten sind,
- die Rolle der Gradientenmatrix $\mathbf{M}$ bei Harris Corners beschreiben,
- die Eigenwerte von $\mathbf{M}$ qualitativ als Fläche, Kante oder Ecke interpretieren,
- Harris Corners und Good Features to Track mit OpenCV anwenden,
- zwischen **Feature** und **Deskriptor** unterscheiden,
- die Grundidee von SIFT und der Verarbeitung über unterschiedliche Skalierungen erklären,
- SIFT-Features mit OpenCV extrahieren,
- Deskriptoren über die euklidische Distanz vergleichen,
- Feature Matches zwischen zwei Bildern bestimmen und visualisieren,
- Grenzen klassischer, von Hand definierter Merkmale einschätzen.

> **Hinweis:** Ziel ist nicht, bereits alle mathematischen Details von Harris und SIFT vollständig zu beherrschen. Das Notebook soll die zentralen Konzepte vor der Vorlesung praktisch erfahrbar machen.

### Hinweise zur Verwendung dieses Notebooks

Dieses Notebook wird über **MyBinder** direkt im Browser ausgeführt. Der vorhandene C++-Code kann unmittelbar in den Notebook-Zellen bearbeitet und ausgeführt werden; eine zusätzliche lokale Installation ist dafür nicht erforderlich.

Das Notebook besteht aus Text- und Codezellen. Die Textzellen enthalten kurze Erklärungen und Aufgabenstellungen, die C++-Codezellen können direkt ausgeführt und verändert werden.

Eine Codezelle wird über den **Play-Button** oder mit **Shift + Enter** ausgeführt. Die Ausgabe erscheint anschließend direkt unterhalb der jeweiligen Zelle.

Arbeiten Sie das Notebook am besten **von oben nach unten** durch, da spätere Codezellen teilweise auf zuvor definierten Variablen und Funktionen aufbauen.

Bei Aufgaben mit `TODO` sollen Sie den vorhandenen Code selbstständig ergänzen oder verändern.

C++ wird mit **xeus-cling** inkrementell ausgeführt. Dadurch können bereits deklarierte Variablen bei einer erneuten Ausführung derselben Zelle zu einer Fehlermeldung führen. Wo dies für eine Übungszelle relevant ist, werden lokale Lambda-Funktionen verwendet. Falls nötig, starten Sie den Kernel neu und führen Sie die Zellen erneut von oben nach unten aus.

> **Wichtig:** Änderungen innerhalb einer Binder-Sitzung werden nicht dauerhaft im GitHub-Repository gespeichert.

## 0. Setup

Die Binder-Umgebung ist bereits mit **xeus-cling** und **OpenCV 4.6** vorbereitet.

> **Binder-/Notebook-Hinweis:** Die folgenden `#pragma cling`-Anweisungen sind ein Workaround für den interaktiven C++-Kernel. Sie teilen `xeus-cling` mit, wo die OpenCV-Header und die kompilierten Bibliotheken liegen. In einem normalen lokalen C++-Projekt wird OpenCV stattdessen beim Kompilieren bzw. über das Build-System, z.B. mit CMake, eingebunden und gelinkt. Im eigentlichen C++-Quellcode genügt dann üblicherweise `#include <opencv2/opencv.hpp>`.

In [ ]:
#include <iostream>
#include <iomanip>
#include <vector>
#include <string>
#include <cmath>
#include <fstream>
#include <cstdlib>
#include <algorithm>

// OpenCV-Pfade für Binder / Ubuntu
#pragma cling add_include_path("/usr/include/opencv4")
#pragma cling add_library_path("/usr/lib/x86_64-linux-gnu")

// Benötigte OpenCV-Bibliotheken laden
#pragma cling load("opencv_core")
#pragma cling load("opencv_imgproc")
#pragma cling load("opencv_imgcodecs")
#pragma cling load("opencv_features2d")
#pragma cling load("opencv_calib3d")

#include <opencv2/opencv.hpp>

std::cout << "OpenCV-Version: " << CV_VERSION << std::endl;
std::cout << "C++ Standard: " << __cplusplus << std::endl;

### Hilfsfunktion zur Darstellung

OpenCV verwendet in einem lokalen C++-Programm normalerweise `cv::imshow()` zur Bilddarstellung. Dabei wird ein eigenes Fenster geöffnet, z.B.

```cpp
cv::imshow("Bild", image);
cv::waitKey(0);
```

In Binder läuft das Notebook jedoch im Browser und besitzt keine normale Desktop-GUI.

> **Binder-/Notebook-Hinweis:** Die folgende Hilfsfunktion ist daher ein Workaround für die Browser-Umgebung. Sie kodiert eine `cv::Mat` als PNG und übergibt sie an die Rich-Display-Funktion des Jupyter-Kernels. Damit ein Bild dargestellt wird, muss `show_image(...)` als **letzte Expression einer Zelle ohne Semikolon** stehen.

In [ ]:
#include "nlohmann/json.hpp"
#include "xtl/xbase64.hpp"

namespace nl = nlohmann;

namespace mvsr
{
    struct NotebookImage
    {
        std::string png_data;

        explicit NotebookImage(const cv::Mat& image)
        {
            std::vector<unsigned char> buffer;
            cv::imencode(".png", image, buffer);

            png_data.assign(
                reinterpret_cast<const char*>(buffer.data()),
                buffer.size()
            );
        }
    };

    nl::json mime_bundle_repr(const NotebookImage& image)
    {
        auto bundle = nl::json::object();
        bundle["image/png"] = xtl::base64encode(image.png_data);
        return bundle;
    }
}

mvsr::NotebookImage show_image(const cv::Mat& image)
{
    return mvsr::NotebookImage(image);
}

### Hilfsfunktion zum Laden externer Testbilder

Einige der folgenden Beispiele verwenden offizielle OpenCV-Testbilder aus GitHub. In der Python-/Colab-Version werden diese mit `!wget` geladen.

> **Binder-/Notebook-Hinweis:** In einer C++-Zelle steht die IPython-Schreibweise `!wget` nicht zur Verfügung. Deshalb wird der Download hier über `std::system()` und das Kommandozeilenprogramm `wget` ausgeführt. In einem normalen lokalen C++-Projekt würde man die benötigten Testbilder üblicherweise lokal ablegen und anschließend direkt mit `cv::imread()` laden.

In [ ]:
bool download_file(
    const std::string& url,
    const std::string& output_file
)
{
    std::string command =
        "wget -q \"" + url + "\" -O \"" + output_file + "\"";

    int result = std::system(command.c_str());

    if (result != 0)
    {
        std::cerr << "Download fehlgeschlagen: "
                  << url << std::endl;
        return false;
    }

    return true;
}

## 1. Testbild laden

Wie in den vorherigen Vorbereitungen verwenden wir zunächst ein **reales Testbild** aus dem GitHub-Repository der Lehrveranstaltung.

Die niedrige Auflösung reicht für die ersten Versuche mit Harris Corners und Good Features to Track aus. Alternativ können Sie auch eigene Bilder im Repository bzw. in der Binder-Sitzung ablegen. Bei anderen Testbildern müssen die Parameter gegebenenfalls angepasst werden.

> **Binder-/Notebook-Hinweis:** MyBinder klont beim Start das vollständige GitHub-Repository. Das Testbild kann daher direkt aus dem Ordner `test_images` geladen werden. Da das Notebook selbst in einem Unterordner liegen kann, werden mehrere mögliche relative Pfade geprüft.

In [ ]:
cv::Mat image;
std::string image_path;

// Niedrige Auflösung
std::string image_filename = "fhtw_logo_low_res.png";

// Alternativ: hohe Auflösung
// std::string image_filename = "fhtw_logo.png";

std::vector<std::string> possible_paths = {
    "test_images/" + image_filename,
    "../test_images/" + image_filename,
    "../../test_images/" + image_filename,
    "../../../test_images/" + image_filename
};

for (const auto& path : possible_paths)
{
    if (std::ifstream(path).good())
    {
        image = cv::imread(path, cv::IMREAD_COLOR);
        image_path = path;
        break;
    }
}

if (image.empty())
{
    std::cerr << "Das Testbild konnte nicht gefunden werden." << std::endl;
}
else
{
    std::cout << "Geladen: " << image_path << std::endl;
    std::cout << "Bildauflösung (H,W,C): "
              << image.rows << ", "
              << image.cols << ", "
              << image.channels() << std::endl;
}

cv::Mat gray;

if (!image.empty())
{
    cv::cvtColor(
        image,
        gray,
        cv::COLOR_BGR2GRAY
    );
}

In [ ]:
show_image(image)

## 2. Von Kanten zu Ecken

In der vorherigen Einheit haben wir Kanten als starke lokale Änderungen der Bildintensität betrachtet. Für viele Aufgaben sind einzelne Kanten jedoch nicht eindeutig genug:

- entlang einer Kante sehen viele lokale Bildbereiche ähnlich aus,
- eine **Ecke** enthält dagegen Änderungen in mehr als einer Richtung.

Die Grundidee von Harris Corners lautet daher:

> **Ein Bildbereich ist besonders interessant, wenn sich sein Aussehen bei einer kleinen Verschiebung in unterschiedliche Richtungen deutlich verändert.**

Für ein Fenster $W$ betrachten wir die Änderung

$f(\Delta x,\Delta y)=
\sum_{x',y' \in W}
\left(
I(x',y')-
I(x'+\Delta x,y'+\Delta y)
\right)^2$.

Mit einer Näherung über die lokalen Bildgradienten lässt sich diese Änderung in Matrixform schreiben:

$f(\Delta x,\Delta y)
\approx
\begin{pmatrix}
\Delta x & \Delta y
\end{pmatrix}
\mathbf{M}
\begin{pmatrix}
\Delta x\\
\Delta y
\end{pmatrix}$

mit

$\mathbf{M}=
\begin{bmatrix}
\sum I_x^2 & \sum I_x I_y\\
\sum I_x I_y & \sum I_y^2
\end{bmatrix}$.

Die beiden Eigenwerte $\lambda_1$ und $\lambda_2$ von $\mathbf{M}$ beschreiben, wie stark sich der lokale Bildbereich in unterschiedlichen Richtungen verändert:

- **beide klein** $\rightarrow$ weitgehend gleichmäßige Fläche,
- **einer groß, einer klein** $\rightarrow$ Kante,
- **beide groß** $\rightarrow$ Ecke.

Damit wird die Eckerkennung unabhängig davon, wie die Ecke im Bild gedreht ist.

In [ ]:
cv::Mat gx;
cv::Mat gy;

cv::Sobel(
    gray,
    gx,
    CV_32F,
    1, 0,
    3
);

cv::Sobel(
    gray,
    gy,
    CV_32F,
    0, 1,
    3
);

cv::Mat gx_abs;
cv::Mat gy_abs;

cv::absdiff(
    gx,
    cv::Scalar::all(0),
    gx_abs
);

cv::absdiff(
    gy,
    cv::Scalar::all(0),
    gy_abs
);

cv::Mat gx_vis;
cv::Mat gy_vis;

cv::normalize(
    gx_abs,
    gx_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

cv::normalize(
    gy_abs,
    gy_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

**Original**

In [ ]:
show_image(image)

**Gradient in x-Richtung**

In [ ]:
show_image(gx_vis)

**Gradient in y-Richtung**

In [ ]:
show_image(gy_vis)

### Beobachtung

Für eine Kante ist typischerweise nur in einer Richtung eine starke Intensitätsänderung vorhanden. An einer Ecke treten dagegen starke Änderungen in mehreren Richtungen auf.

Genau diese lokale Information wird von Harris über die Matrix $\mathbf{M}$ zusammengefasst.

> **Binder-/Notebook-Hinweis:** In der Python-Version werden mehrere Bilder mit Matplotlib direkt nebeneinander dargestellt. Da in diesem C++-Notebook die browserbasierte Rich-Display-Ausgabe verwendet wird, werden sie in separaten Zellen angezeigt. In einem lokalen C++-Programm könnten dafür mehrere `cv::imshow()`-Fenster verwendet werden.

## 3. Harris Corners

OpenCV stellt die Harris-Eckerkennung mit `cv::cornerHarris()` bereit.

Wichtige Parameter sind:

- **`blockSize`** – Größe des betrachteten Nachbarschaftsbereichs,
- **`ksize`** – Kernelgröße für die Sobel-Ableitungen,
- **`k`** – Harris-Parameter für die Bewertung der lokalen Struktur.

Das Ergebnis ist zunächst keine Liste von Eckpunkten, sondern eine **Antwortkarte**: hohe Werte entsprechen Positionen, die stark auf das Harris-Kriterium reagieren.

In [ ]:
cv::Mat gray_harris;

gray.convertTo(
    gray_harris,
    CV_32F
);

cv::Mat harris_response;

cv::cornerHarris(
    gray_harris,
    harris_response,
    2,
    3,
    0.04
);

// Für die Darstellung werden lokale Maxima etwas vergrößert
cv::Mat harris_response_dilated;

cv::dilate(
    harris_response,
    harris_response_dilated,
    cv::Mat()
);

cv::Mat harris_vis;

cv::normalize(
    harris_response_dilated,
    harris_vis,
    0, 255,
    cv::NORM_MINMAX,
    CV_8U
);

cv::Mat harris_vis_color;

cv::applyColorMap(
    harris_vis,
    harris_vis_color,
    cv::COLORMAP_VIRIDIS
);

double harris_min_value;
double harris_max_value;

cv::minMaxLoc(
    harris_response_dilated,
    &harris_min_value,
    &harris_max_value
);

double threshold_harris =
    0.01 * harris_max_value;

cv::Mat image_harris =
    image.clone();

cv::Mat harris_mask =
    harris_response_dilated > threshold_harris;

image_harris.setTo(
    cv::Scalar(0, 0, 255),
    harris_mask
);

**Original**

In [ ]:
show_image(image)

**Harris-Antwort**

In [ ]:
show_image(harris_vis_color)

**Gefundene Eckbereiche**

In [ ]:
show_image(image_harris)

### Mini-Aufgabe: Harris-Parameter

Verändern Sie die Parameter der Harris-Eckerkennung und beobachten Sie, welche Bildbereiche als Ecken markiert werden.

Weitere Informationen finden Sie in der [OpenCV-4.6-Dokumentation zu `cornerHarris()`](https://docs.opencv.org/4.6.0/dd/d1a/group__imgproc__feature.html).

Beobachten Sie insbesondere:

- Wie verändert `blockSize` die lokale Betrachtung?
- Welchen Einfluss hat der Threshold auf die Anzahl der markierten Eckbereiche?
- Welche Parameter führen zu einer sinnvollen Auswahl im Testbild?

> **Binder-/Notebook-Hinweis:** Die Übungszelle wird in einer Lambda-Funktion gekapselt. Dadurch bleiben die Testvariablen lokal und die Zelle kann nach Änderungen leichter erneut ausgeführt werden. In einem normalen lokalen C++-Programm wäre diese zusätzliche Lambda-Konstruktion nicht notwendig.

In [ ]:
[]()
{
    // TODO: Parameter der Harris-Eckerkennung verändern

    int block_size_test = 2;
    int ksize_test = 3;
    double k_test = 0.04;
    double threshold_rel_test = 0.01;

    cv::Mat harris_test;

    cv::cornerHarris(
        gray_harris,
        harris_test,
        block_size_test,
        ksize_test,
        k_test
    );

    cv::dilate(
        harris_test,
        harris_test,
        cv::Mat()
    );

    double min_value;
    double max_value;

    cv::minMaxLoc(
        harris_test,
        &min_value,
        &max_value
    );

    cv::Mat image_harris_test =
        image.clone();

    cv::Mat mask =
        harris_test > threshold_rel_test * max_value;

    image_harris_test.setTo(
        cv::Scalar(0, 0, 255),
        mask
    );

    std::cout
        << "Harris: blockSize=" << block_size_test
        << ", ksize=" << ksize_test
        << ", k=" << k_test
        << ", threshold=" << threshold_rel_test
        << std::endl;

    return show_image(image_harris_test);
}()

## 4. Good Features to Track

Harris liefert eine Antwort für sehr viele Bildpositionen. Für Anwendungen wie Tracking ist häufig eine **begrenzte Anzahl gut lokalisierter Eckpunkte** praktischer.

[**Good Features to Track**](https://docs.opencv.org/4.6.0/dd/d1a/group__imgproc__feature.html) nach Shi und Tomasi erweitert die Eckerkennung durch eine andere Bewertung der Eigenwerte. OpenCV stellt das Verfahren mit `cv::goodFeaturesToTrack()` bereit.

Die Funktion liefert direkt die Koordinaten ausgewählter Eckpunkte.

In [ ]:
std::vector<cv::Point2f> corners;

cv::goodFeaturesToTrack(
    gray,
    corners,
    50,
    0.01,
    7,
    cv::Mat(),
    3
);

cv::Mat image_gftt =
    image.clone();

for (const auto& corner : corners)
{
    cv::circle(
        image_gftt,
        corner,
        3,
        cv::Scalar(0, 0, 255),
        -1
    );
}

std::cout << "Gefundene Eckpunkte: "
          << corners.size() << std::endl;

In [ ]:
show_image(image_gftt)

### Mini-Aufgabe: Auswahl der Eckpunkte

Die wichtigsten Parameter sind:

- **`maxCorners`** – maximale Anzahl zurückgegebener Eckpunkte,
- **`qualityLevel`** – Mindestqualität relativ zum besten gefundenen Punkt,
- **`minDistance`** – minimaler Abstand zwischen zwei ausgewählten Eckpunkten.

Verändern Sie die Parameter und beobachten Sie, wie sich Anzahl und Verteilung der Features ändern.

In [ ]:
[]()
{
    // TODO: Parameter von Good Features to Track verändern

    int max_corners_test = 30;
    double quality_level_test = 0.05;
    double min_distance_test = 10;

    std::vector<cv::Point2f> corners_test;

    cv::goodFeaturesToTrack(
        gray,
        corners_test,
        max_corners_test,
        quality_level_test,
        min_distance_test,
        cv::Mat(),
        3
    );

    cv::Mat image_gftt_test =
        image.clone();

    for (const auto& corner : corners_test)
    {
        cv::circle(
            image_gftt_test,
            corner,
            3,
            cv::Scalar(0, 0, 255),
            -1
        );
    }

    std::cout << "Gefundene Eckpunkte: "
              << corners_test.size() << std::endl;

    return show_image(image_gftt_test);
}()

## 5. Von Features zu Deskriptoren

Mit Harris oder Good Features to Track erhalten wir zunächst ein Set markanter **Bildpositionen**.

Für Tracking können solche Eckpunkte bereits verwendet werden. Für die Objekterkennung reicht die Position einer Ecke allein jedoch nicht aus.

Wir unterscheiden daher:

- **Feature / Keypoint:** markante Position im Bild,
- **Deskriptor:** numerische Beschreibung der lokalen Umgebung dieses Features.

Eine sehr einfache Idee für einen Deskriptor wäre, ein kleines Pixel-Fenster um einen Eckpunkt direkt zu speichern. Das ist jedoch empfindlich gegenüber Rotation, Skalierung, Beleuchtung und anderen Veränderungen.

In [ ]:
int patch_radius = 10;

std::vector<cv::Point> valid_corners;

for (const auto& corner : corners)
{
    int x = cvRound(corner.x);
    int y = cvRound(corner.y);

    if (
        x - patch_radius >= 0 &&
        y - patch_radius >= 0 &&
        x + patch_radius < gray.cols &&
        y + patch_radius < gray.rows
    )
    {
        valid_corners.emplace_back(x, y);
    }
}

if (valid_corners.empty())
{
    std::cerr << "Kein geeigneter Eckpunkt für den Patch gefunden."
              << std::endl;
}

cv::Point selected_corner =
    valid_corners.front();

cv::Rect patch_roi(
    selected_corner.x - patch_radius,
    selected_corner.y - patch_radius,
    2 * patch_radius + 1,
    2 * patch_radius + 1
);

cv::Mat patch =
    gray(patch_roi).clone();

cv::Mat image_patch =
    image.clone();

cv::rectangle(
    image_patch,
    patch_roi,
    cv::Scalar(0, 0, 255),
    2
);

**Feature und lokales Fenster**

In [ ]:
show_image(image_patch)

**Einfacher Patch als Beschreibung**

In [ ]:
show_image(patch)

### Problem einfacher Patches

Ein direkt gespeichertes Pixel-Fenster ist stark an die konkrete Darstellung im Bild gebunden.

Wird das Objekt gedreht oder erscheint es in einer anderen Größe, verändert sich auch der Patch deutlich.

Für eine robustere Objekterkennung benötigen wir daher

1. geeignete Features auch bei unterschiedlichen **Skalierungen** und
2. Deskriptoren, die die lokale Struktur charakteristischer beschreiben.

Damit kommen wir zu **SIFT**.

## 6. SIFT – Scale Invariant Feature Transform

SIFT kombiniert zwei Aufgaben:

- **Detektion** markanter Features,
- Berechnung eines **Deskriptors** für jedes Feature.

Die zentrale Idee ist, relevante Punkte nicht nur in einem einzelnen Bildmaßstab zu suchen.

Dafür werden Bilder in unterschiedlichen Skalierungen betrachtet und mehrfach geglättet. Unterschiede zwischen benachbarten geglätteten Bildern helfen dabei, lokale Extremstellen über verschiedene Skalen zu finden.

Für verbleibende Features wird zusätzlich die lokale Gradienteninformation betrachtet. Dadurch kann eine charakteristische Orientierung bestimmt werden.

Der SIFT-Deskriptor fasst die lokalen Gradientenrichtungen in **4 × 4 Histogrammen mit jeweils 8 Bins** zusammen:

$4 \cdot 4 \cdot 8 = 128$

Jedes SIFT-Feature besitzt damit einen **128-dimensionalen Deskriptor**.

Für die folgenden Versuche verwenden wir die offiziellen OpenCV-Testbilder `box.png` und `box_in_scene.png`.

Das erste Bild zeigt das gesuchte Objekt. Im zweiten Bild befindet sich dasselbe Objekt in einer größeren Szene und aus einer veränderten Ansicht.

In [ ]:
download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/box.png",
    "box.png"
);

download_file(
    "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/box_in_scene.png",
    "box_in_scene.png"
);

cv::Mat image_object =
    cv::imread("box.png", cv::IMREAD_COLOR);

cv::Mat image_scene =
    cv::imread("box_in_scene.png", cv::IMREAD_COLOR);

if (image_object.empty())
{
    std::cerr << "box.png konnte nicht geladen werden."
              << std::endl;
}

if (image_scene.empty())
{
    std::cerr << "box_in_scene.png konnte nicht geladen werden."
              << std::endl;
}

cv::Mat gray_object;
cv::Mat gray_scene;

cv::cvtColor(
    image_object,
    gray_object,
    cv::COLOR_BGR2GRAY
);

cv::cvtColor(
    image_scene,
    gray_scene,
    cv::COLOR_BGR2GRAY
);

**Objektbild**

In [ ]:
show_image(image_object)

**Szenenbild**

In [ ]:
show_image(image_scene)

## 7. SIFT-Features und Deskriptoren mit OpenCV

Mit `cv::SIFT::create()` erzeugen wir einen SIFT-Detektor. `detectAndCompute()` liefert anschließend

- eine Liste der gefundenen **Keypoints** und
- eine Matrix mit den zugehörigen **Deskriptoren**.

Bei der Darstellung mit `DRAW_RICH_KEYPOINTS` codiert die Kreisgröße die gefundene Skalierung des Features. Zusätzlich wird seine Orientierung dargestellt.

In [ ]:
cv::Ptr<cv::SIFT> sift =
    cv::SIFT::create();

std::vector<cv::KeyPoint> keypoints_object;
cv::Mat descriptors_object;

sift->detectAndCompute(
    gray_object,
    cv::noArray(),
    keypoints_object,
    descriptors_object
);

cv::Mat image_sift;

cv::drawKeypoints(
    image_object,
    keypoints_object,
    image_sift,
    cv::Scalar::all(-1),
    cv::DrawMatchesFlags::DRAW_RICH_KEYPOINTS
);

std::cout << "Anzahl SIFT-Features: "
          << keypoints_object.size() << std::endl;

std::cout << "Form der Deskriptor-Matrix: "
          << descriptors_object.rows << " x "
          << descriptors_object.cols << std::endl;

In [ ]:
show_image(image_sift)

Die Deskriptor-Matrix besitzt die Form

`(Anzahl Features, 128)`.

Jede Zeile beschreibt damit genau ein gefundenes Feature durch einen 128-dimensionalen Vektor.

Wir betrachten beispielhaft das erste Feature.

In [ ]:
const cv::KeyPoint& first_keypoint =
    keypoints_object.front();

cv::Mat first_descriptor =
    descriptors_object.row(0);

std::cout << "Position: ("
          << first_keypoint.pt.x << ", "
          << first_keypoint.pt.y << ")" << std::endl;

std::cout << "Skalierung / Größe: "
          << first_keypoint.size << std::endl;

std::cout << "Orientierung: "
          << first_keypoint.angle
          << " Grad" << std::endl;

std::cout << "Descriptor-Dimension: "
          << first_descriptor.cols << std::endl;

std::cout << "Erste 16 Descriptor-Werte:"
          << std::endl;

for (int i = 0; i < std::min(16, first_descriptor.cols); ++i)
{
    std::cout
        << first_descriptor.at<float>(0, i)
        << (i < 15 ? " " : "\n");
}

### Mini-Aufgabe: SIFT-Detektion

OpenCV erlaubt unter anderem, die maximale Anzahl gewünschter Features sowie einen Kontrast-Schwellwert einzustellen.

Verändern Sie die Parameter und beobachten Sie:

- Wie verändert sich die Anzahl der gefundenen Features?
- Welche Bildbereiche bleiben bei einem strengeren `contrastThreshold` übrig?
- Sind alle gefundenen Features gleich charakteristisch?

> **Binder-/Notebook-Hinweis:** Die SIFT-Übungszelle wird wieder lokal in einer Lambda-Funktion ausgeführt, damit die Testparameter mehrfach verändert werden können, ohne globale Variablen neu zu deklarieren.

In [ ]:
[]()
{
    // TODO: SIFT-Parameter verändern

    int nfeatures_test = 0;
    double contrast_threshold_test = 0.04;

    cv::Ptr<cv::SIFT> sift_test =
        cv::SIFT::create(
            nfeatures_test,
            3,
            contrast_threshold_test
        );

    std::vector<cv::KeyPoint> keypoints_test;
    cv::Mat descriptors_test;

    sift_test->detectAndCompute(
        gray_object,
        cv::noArray(),
        keypoints_test,
        descriptors_test
    );

    cv::Mat image_sift_test;

    cv::drawKeypoints(
        image_object,
        keypoints_test,
        image_sift_test,
        cv::Scalar::all(-1),
        cv::DrawMatchesFlags::DRAW_RICH_KEYPOINTS
    );

    std::cout << "Gefundene SIFT-Features: "
              << keypoints_test.size() << std::endl;

    std::cout << "contrastThreshold: "
              << contrast_threshold_test << std::endl;

    return show_image(image_sift_test);
}()

## 8. Feature Matching

Für die Objekterkennung müssen die Deskriptoren des bekannten Objekts mit den Deskriptoren eines neuen Bildes verglichen werden.

Für zwei Deskriptoren $\vec{x}$ und $\vec{y}$ kann dazu die **euklidische Distanz** verwendet werden:

$ED(\vec{x},\vec{y})
=
\|\vec{x}-\vec{y}\|_2
=
\sqrt{
\sum_{i=1}^{d}(x_i-y_i)^2
}$.

Für SIFT gilt $d=128$.

Je **kleiner** die Distanz, desto ähnlicher sind sich zwei Deskriptoren.

OpenCV bezeichnet diese Distanz als `NORM_L2`. Für einen einfachen Vergleich verwenden wir hier einen `cv::BFMatcher` (**Brute Force Matcher**), der Deskriptoren direkt miteinander vergleicht.

In [ ]:
std::vector<cv::KeyPoint> keypoints_scene;
cv::Mat descriptors_scene;

sift->detectAndCompute(
    gray_scene,
    cv::noArray(),
    keypoints_scene,
    descriptors_scene
);

cv::BFMatcher matcher(
    cv::NORM_L2,
    true
);

std::vector<cv::DMatch> matches;

matcher.match(
    descriptors_object,
    descriptors_scene,
    matches
);

std::sort(
    matches.begin(),
    matches.end(),
    [](const cv::DMatch& a, const cv::DMatch& b)
    {
        return a.distance < b.distance;
    }
);

std::cout << "Features im Objektbild: "
          << keypoints_object.size() << std::endl;

std::cout << "Features im Szenenbild: "
          << keypoints_scene.size() << std::endl;

std::cout << "Gefundene Matches: "
          << matches.size() << std::endl;

std::cout << "Kleinste Distanzen:"
          << std::endl;

for (std::size_t i = 0;
     i < std::min<std::size_t>(5, matches.size());
     ++i)
{
    std::cout << std::fixed
              << std::setprecision(2)
              << matches[i].distance
              << std::endl;
}

### Distanz eines Matches nachvollziehen

Der Matcher liefert für jedes Match bereits eine Distanz. Für das beste Match berechnen wir die euklidische Distanz zusätzlich direkt mit OpenCV.

Beide Werte sollten übereinstimmen.

In [ ]:
cv::DMatch best_match =
    matches.front();

cv::Mat descriptor_object =
    descriptors_object.row(
        best_match.queryIdx
    );

cv::Mat descriptor_scene =
    descriptors_scene.row(
        best_match.trainIdx
    );

double distance_manual =
    cv::norm(
        descriptor_object,
        descriptor_scene,
        cv::NORM_L2
    );

std::cout << "Distanz laut OpenCV: "
          << best_match.distance << std::endl;

std::cout << "Manuell berechnete L2-Distanz: "
          << distance_manual << std::endl;

## 9. Matches visualisieren

Die besten Matches werden als Linien zwischen Objektbild und Szenenbild dargestellt.

Damit wird sichtbar, welche lokalen Merkmale des bekannten Objekts im neuen Bild wiedergefunden wurden.

In [ ]:
int num_matches = 30;

std::vector<cv::DMatch> best_matches(
    matches.begin(),
    matches.begin() +
        std::min<int>(
            num_matches,
            static_cast<int>(matches.size())
        )
);

cv::Mat image_matches;

cv::drawMatches(
    image_object,
    keypoints_object,
    image_scene,
    keypoints_scene,
    best_matches,
    image_matches,
    cv::Scalar::all(-1),
    cv::Scalar::all(-1),
    std::vector<char>(),
    cv::DrawMatchesFlags::NOT_DRAW_SINGLE_POINTS
);

std::cout << "Dargestellte Matches: "
          << best_matches.size() << std::endl;

In [ ]:
show_image(image_matches)

### Mini-Aufgabe: Match-Qualität

Die kleinste Distanz ist jeweils der ähnlichste gefundene Deskriptor. Das bedeutet jedoch nicht automatisch, dass jedes Match korrekt ist.

Verwenden Sie einen Distanz-Schwellwert und beobachten Sie, welche Matches übrig bleiben.

- Was passiert bei einem sehr großen Threshold?
- Was passiert bei einem sehr kleinen Threshold?
- Sind die kleinsten Distanzen immer geometrisch plausibel?

In [ ]:
[]()
{
    // TODO: Distanz-Schwellwert und Anzahl dargestellter Matches verändern

    float distance_threshold_test = 200.0f;
    int max_matches_test = 30;

    std::vector<cv::DMatch> matches_test;

    for (const auto& match : matches)
    {
        if (match.distance < distance_threshold_test)
        {
            matches_test.push_back(match);
        }

        if (
            static_cast<int>(matches_test.size())
            >= max_matches_test
        )
        {
            break;
        }
    }

    std::cout << "Matches unterhalb des Thresholds: "
              << matches_test.size() << std::endl;

    cv::Mat image_matches_test;

    cv::drawMatches(
        image_object,
        keypoints_object,
        image_scene,
        keypoints_scene,
        matches_test,
        image_matches_test,
        cv::Scalar::all(-1),
        cv::Scalar::all(-1),
        std::vector<char>(),
        cv::DrawMatchesFlags::NOT_DRAW_SINGLE_POINTS
    );

    return show_image(image_matches_test);
}()

## 10. Warum reicht das nicht immer?

Mit SIFT verfügen wir über deutlich aussagekräftigere lokale Merkmale als bei einem einfachen Pixel-Patch.

Trotzdem bleibt klassische, von Hand definierte Merkmalserkennung begrenzt:

- die Qualität hängt davon ab, ob geeignete Features gefunden werden,
- Deskriptordistanzen sind nur ein Ähnlichkeitsmaß und können falsche Matches liefern,
- starke Perspektiv- und 3D-Veränderungen können problematisch werden,
- reale Szenen enthalten unterschiedliche Beleuchtung, Schatten, Verdeckungen und mehrdeutige Strukturen.

Damit entsteht die Motivation für **gelernte Merkmale**:

> Statt die Regeln für ein gutes Merkmal vollständig von Hand vorzugeben, können charakteristische Merkmale anhand von Daten gelernt werden.

Dieser Übergang führt zu statistischen Verfahren sowie Machine- und Deep-Learning-Ansätzen.

## 11. Selbstcheck

Beantworten Sie die Fragen zunächst ohne in die Vorlesungsunterlagen zu schauen.

1. Warum ist eine Ecke häufig eindeutiger als eine einzelne Kante?
2. Welche Information wird in der Matrix $\mathbf{M}$ von Harris zusammengefasst?
3. Wie werden die beiden Eigenwerte qualitativ für Fläche, Kante und Ecke interpretiert?
4. Was ist der Unterschied zwischen einem Feature und einem Deskriptor?
5. Warum ist ein direkt gespeicherter Pixel-Patch als Deskriptor nur begrenzt robust?
6. Welche beiden Aufgaben kombiniert SIFT?
7. Warum betrachtet SIFT unterschiedliche Skalierungen eines Bildes?
8. Wie lang ist ein SIFT-Deskriptor?
9. Welche Distanz verwenden wir hier zum Vergleich von SIFT-Deskriptoren?
10. Was bedeutet eine kleine Deskriptordistanz?
11. Warum reicht der kleinste Abstand allein noch nicht aus, um jedes Match als korrekt zu betrachten?
12. Warum motivieren klassische Features den Übergang zu gelernten Merkmalen?

<details>
<summary><b>Kurze Antworten anzeigen</b></summary>

1. An einer Ecke ändert sich die Bildintensität in mehreren Richtungen, wodurch die Position lokaler eindeutiger wird.
2. Die lokalen Bildgradienten und ihre gemeinsame Variation innerhalb eines Fensters.
3. Beide klein: Fläche; einer groß und einer klein: Kante; beide groß: Ecke.
4. Ein Feature ist eine markante Bildposition; ein Deskriptor beschreibt ihre lokale Umgebung numerisch.
5. Der Pixelinhalt verändert sich stark bei Rotation, Skalierung, Beleuchtung oder anderen Bildänderungen.
6. Feature-Detektion und Berechnung von Deskriptoren.
7. Damit markante Strukturen auch bei unterschiedlichen Objektgrößen bzw. Entfernungen gefunden werden können.
8. 128 Werte.
9. Die euklidische Distanz bzw. L2-Distanz.
10. Die beiden Deskriptoren sind im Deskriptorraum ähnlich.
11. Auch ein falscher Kandidat kann der ähnlichste verfügbare Deskriptor sein.
12. Reale Objekte und Szenen sind oft zu komplex, um alle relevanten Merkmale und Variationen zuverlässig durch feste Regeln abzudecken.

</details>

## 12. Take-away

Für die Präsenz-LV sollten Sie folgende Punkte mitnehmen:

- Ecken sind lokale Bildbereiche mit starken Änderungen in mehreren Richtungen.
- Harris analysiert lokale Gradienten über eine Matrix $\mathbf{M}$ und deren Eigenwerte.
- Good Features to Track liefert eine Auswahl gut lokalisierter Eckpunkte.
- Ein **Feature** bezeichnet eine markante Bildposition; ein **Deskriptor** beschreibt die Umgebung dieses Features.
- Ein einfacher Pixel-Patch ist gegenüber Bildveränderungen nur wenig robust.
- SIFT sucht Features über unterschiedliche Skalierungen und berücksichtigt lokale Gradientenrichtungen.
- Ein SIFT-Deskriptor besitzt **128 Dimensionen**.
- SIFT-Deskriptoren können über die **euklidische Distanz (L2)** verglichen werden.
- Feature Matching stellt Korrespondenzen zwischen lokalen Merkmalen zweier Bilder her.
- Auch gute klassische Merkmale bleiben bei komplexen realen Szenen begrenzt.

### Ausblick

In der Vorlesung werden Harris, Good Features to Track und SIFT systematisch hergeleitet und eingeordnet. Die Grenzen handdefinierter Merkmale bilden anschließend den Übergang zu **gelernten Bildmerkmalen**.